# Design of Symmetry-Resolved Real-Space Exact Diagonalization (Python)

**Abstract.** This notebook is a self-contained derivation of the *symmetry-resolved exact
diagonalization* (ED) engine inside `realspace_exactdiagonalization_py`, the Python port of
`RealSpace_ExactDiagonalization.jl`. The engine diagonalizes interacting hard-core-boson and
spinless-fermion lattice models on **arbitrary real-space graphs** by (i) encoding Fock states as
bitmasks $m=\sum_i n_i 2^{i-1}$, (ii) partitioning the Hilbert space into symmetry orbits with a
canonical representative and its stabilizer data, and (iii) projecting onto one-dimensional
irreducible representations with explicit $\sqrt{|\mathrm{Stab}|}$ normalization. It supports a
*matrix* mode (explicit sparse block) and a *matrix-free* mode (on-the-fly $H|\psi\rangle$), and it
handles flux-threaded boundary conditions through a **flux-aware translation group**. The design
follows the orbit–stabilizer philosophy of [XDiag](https://github.com/awietek/xdiag), with the
twists that everything is expressed in bitmask arithmetic and that the whole pipeline is
statistics-agnostic (bosonic vs fermionic).

**References.**
1. A. Wietek & A. Läuchli, *XDiag* — a symmetry-resolved exact diagonalization library; [github.com/awietek/xdiag](https://github.com/awietek/xdiag).
2. K. Sun, Z. Gu, H. Katsura, S. Das Sarma, *Nearly flatbands with nontrivial topology*, Phys. Rev. Lett. **106**, 236803 (2011).
3. D. N. Sheng, Z.-C. Gu, K. Sun, L. Sheng, *Fractional Chern insulator on the honeycomb lattice with bosons*, Phys. Rev. Lett. **107**, 146803 (2011).
4. R. Resta, *Quantum-mechanical position operator in extended systems*, Phys. Rev. Lett. **80**, 1800 (1998).
5. The Julia reference package: `RealSpace_ExactDiagonalization.jl` (its `doc/design.ipynb` is the template for this notebook).


## Motivation and Scope

The goal of this package is **symmetry-resolved exact diagonalization** for interacting quantum
lattice models — *both bosonic and fermionic* — on *arbitrary real-space graphs*. The design follows
the standard symmetry-resolved ED of [XDiag](https://github.com/awietek/xdiag), while providing a
self-contained, pedagogical implementation.

The key design ideas are:

1. **Bitmask encoding** — every Fock configuration $|\mathbf s\rangle=|n_1,\ldots,n_N\rangle$
   (hard-core constraint $n_i\in\{0,1\}$) is a single integer $m=\sum_i n_i 2^{i-1}$, enabling
   $O(1)$ bitwise operations.
2. **Orbit–stabilizer decomposition** — the Hilbert space is partitioned into orbits under the
   symmetry group $G$, each labelled by a canonical representative $|[\mathbf s]\rangle$ and its
   stabilizer subgroup.
3. **Irrep-induced projection** — one-dimensional irreducible representations (irreps) of finite
   abelian groups build projectors $P_\chi$ that block-diagonalize $H$ without ever forming the
   full matrix.
4. **Two computational modes** — a *matrix* mode (explicit sparse block, fast, memory-hungry) and a
   *matrix-free* mode (on-the-fly $H|\psi\rangle$ via shared-memory parallel Lanczos, near-zero
   memory).
5. **Unified boson/fermion treatment** — the pipeline is statistics-agnostic; fermionic signs
   (permutation parity + Jordan–Wigner strings) are injected through a `Particle_Statistics` flag.


## Problem Setup

### Many-Body Hilbert Space

Consider $N_e$ **spinless** particles on a graph of $N$ vertices. The vertices represent *all*
internal degrees of freedom flattened into one index — spatial sites, spin, valley, sublattice,
band indices, etc. We work in the occupation basis

\begin{equation}
|\mathbf s\rangle \equiv |n_1,\ldots,n_N\rangle,\qquad n_i\in\{0,1\},\qquad \sum_{i=1}^N n_i = N_e .
\end{equation}

The hard-core constraint $n_i\in\{0,1\}$ is natural for fermions (Pauli exclusion) and a valid
approximation for strongly interacting bosons. The dimension at fixed $N_e$ is $\binom{N}{N_e}$,
which grows combinatorially — the central challenge of ED.

> **⚠️ `filling_fraction` convention.** In `build_ed_data`, `filling_fraction` is **particles per
> *flattened* graph vertex**: $N_e = \mathrm{filling\_fraction}\times N_{\text{vertices}}$. This is
> **not** the "filling per band". For the bosonic Haldane FCI at half band filling on $2\times3$
> unit cells there are $2\times 2\times3 = 12$ vertices and $3$ bosons, so the vertex filling is
> $3/12 = 1/4$ (the band filling $\nu=1/2$). Always count every spin/sublattice/band as a separate
> vertex, then compute $N_e/N_{\text{vertices}}$.


### Hamiltonian

As a running example, consider a hard-core model on a 2D lattice with periodic boundary conditions:

\begin{equation}
H = \sum_{\langle i,j\rangle} t_{ij}\, a_i^\dagger a_j + \sum_{\langle i,j\rangle} V_{ij}\, n_i n_j ,
\end{equation}

where $a_i^\dagger$ is the canonical creation operator ($b_i^\dagger$ for bosons, $c_i^\dagger$ for
fermions). The engine accepts arbitrary short-range terms encoded as lists of
`(from_site, to_site, amplitude)` tuples — a *bilinear* (hopping) list and a *density–density*
list, both with **1-based** site indices.


## Bitmask Representation

### Encoding

The hard-core constraint makes each configuration a binary string of length $N$, encoded as one
unsigned integer:

\begin{equation}
\boxed{|\mathbf s\rangle \equiv |n_1,\ldots,n_N\rangle \;\longmapsto\; m = \sum_{i=1}^N n_i\,2^{i-1} \;\in\;\{0,\ldots,2^N-1\}.}
\end{equation}

Bit $i-1$ (0-based) corresponds to vertex $i$ (1-based). This gives $O(1)$ bitwise operations:

| Operation | Python |
|---|---|
| Occupancy test | `(m & (1 << (i-1))) != 0` |
| Create particle | `m | (1 << (i-1))` |
| Annihilate particle | `m & ~(1 << (i-1))` |
| Count particles | `m.bit_count()` |

The primitives live in `bitwise_operations` (`bitmask_of_site`, `occupy_site_for_mask`,
`empty_site_for_mask`, `is_site_occupied`, `n_occupied_for_mask`, ...).

### Why Bitmasks?

1. **Compact storage**: one integer per configuration ($N\le 63$ for the 64-bit kernels).
2. **Lexicographic order**: integer comparison `m < m'` gives a canonical ordering for orbit
   representatives.
3. **Hardware acceleration**: `popcount`/`ctz` and bitwise AND/OR/XOR are single CPU instructions.
4. **Gosper's-hack compatibility**: fixed particle number $\Leftrightarrow$ fixed Hamming weight.


In [1]:
import realspace_exactdiagonalization_py as ed
from realspace_exactdiagonalization_py import bitwise_operations as bw

# Bitmask encoding: 1-based site i -> bit (i-1)
m = bw.bitmask_of_site(3)           # occupy site 3
m = bw.occupy_site_for_mask(m, 1)   # occupy site 1
print(f"mask = {m} = {bin(m)}")     # 0b101
print("occupied sites:", bw.decode_bit_mask_to_configuration(m, n_site=5))
print("n_occupied:", bw.n_occupied_for_mask(m))
print("site 1 occupied?", bw.is_site_occupied(m, 1), "| site 2 empty?", bw.is_site_empty(m, 2))
print("clear site 1   :", bin(bw.empty_site_for_mask(m, 1)))

# Gosper's hack: enumerate all masks with a fixed number of set bits, in order
def gosper_next(x: int) -> int:
    c = x & -x            # isolate the rightmost 1-bit
    r = x + c             # add it (carry chain)
    return (((r ^ x) >> 2) // c) | r

x = (1 << 3) - 1          # lowest weight-3 mask (0b111)
seen = []
while x < (1 << 6):       # stop at 2**6
    seen.append(x)
    x = gosper_next(x)
print("\nAll 3-of-6 masks via Gosper's hack:")
print([bin(v) for v in seen])
from math import comb
assert len(seen) == comb(6, 3)
print(f"count = {len(seen)} = C(6,3) ✓")


mask = 5 = 0b101
occupied sites: [1, 3]
n_occupied: 2
site 1 occupied? True | site 2 empty? True
clear site 1   : 0b100

All 3-of-6 masks via Gosper's hack:
['0b111', '0b1011', '0b1101', '0b1110', '0b10011', '0b10101', '0b10110', '0b11001', '0b11010', '0b11100', '0b100011', '0b100101', '0b100110', '0b101001', '0b101010', '0b101100', '0b110001', '0b110010', '0b110100', '0b111000']
count = 20 = C(6,3) ✓


## Finite Symmetry Group

### Symmetry Action on Second-Quantized Operators

For a finite system there is no spontaneous symmetry breaking; symmetry means $[H,U_g]=0$ for a
unitary representation $U_g$ of a group $G$. The action on creation operators is

\begin{equation}
\boxed{U_g\, a_i^\dagger\, U_g^{-1} = \eta_g(i)\, a_{\pi_g(i)}^\dagger ,}
\end{equation}

where $\pi_g\in S_N$ is a vertex permutation and $\eta_g(i)\in U(1)$ a site-dependent phase. On a
Fock state (bitmask) this is

\begin{equation}
U_g\,|m\rangle = \alpha_g(m)\,|m'\rangle,\qquad
\alpha_g(m) = \prod_{i\in\mathrm{occ}(m)} \eta_g(i),\qquad
m' = \sum_{i\in\mathrm{occ}(m)} 2^{\pi_g(i)-1}.
\end{equation}

### Data Structures

Each group element is a `Symmetry_Operation(label, perm, perm_phases=...)` (1-based permutation,
complex phases). The group is `Finite_Symmetry_Group(name, operations, identity_idx=0)` — an
**ordered** list of operations. The order is critical: every irrep character vector is indexed by
the *same* order, giving $O(1)$ lookup of $\chi(g_i)$. Note the Python port uses a **0-based**
`identity_idx` (the Julia package uses 1-based).

Typical groups: **identity** ($G=\{e\}$), **2D translations** ($G=\mathbb{Z}_{L_1}\times\mathbb{Z}_{L_2}$), point groups ($C_3,C_4,C_6$), and direct products thereof. (The current package ships
identity + translation irrep builders; non-abelian point groups would need multi-dimensional
projectors, beyond scope.)


In [2]:
import math
import realspace_exactdiagonalization_py as ed
from tightbinding_py import initialize_real_space_lattice

# A [2,3] Haldane-honeycomb lattice (2 sublattices) -> 12 graph vertices
lattice = initialize_real_space_lattice(
    sample_size=[2, 3],
    brav_vec_list=[[1.0, 0.0], [1 / 2, math.sqrt(3) / 2]],
    sub_crys_list=[[0.0, 0.0], [1 / 3, 1 / 3]],
    lattice_name="Haldane_Honeycomb",
    pbc_indicator=[True, True],
)
print("n_site =", lattice.n_site, "| n_sub =", lattice.n_sub, "| sample_size =", lattice.sample_size)

G = ed.build_translation_group(lattice)
print("group:", G)

# Momentum irreps: chi_{(k1,k2)}(d1,d2) = exp(2πi (k1 d1/L1 + k2 d2/L2))
irrep_list = ed.build_irrep_list(G, lattice)
print("|G| =", ed.group_order(G), "| #irreps =", len(irrep_list))
for ir in irrep_list:
    print(f"  irrep {ir.label}: chi = {ir.values.round(3)}")


n_site = 12 | n_sub = 2 | sample_size = [2, 3]
group: Finite_Symmetry_Group(name='translations', n_site=12, |G|=6)
|G| = 6 | #irreps = 6
  irrep (0, 0): chi = [1.+0.j 1.+0.j 1.+0.j 1.+0.j 1.+0.j 1.+0.j]
  irrep (0, 1): chi = [ 1. +0.j    -0.5+0.866j -0.5-0.866j  1. +0.j    -0.5+0.866j -0.5-0.866j]
  irrep (0, 2): chi = [ 1. +0.j    -0.5-0.866j -0.5+0.866j  1. +0.j    -0.5-0.866j -0.5+0.866j]
  irrep (1, 0): chi = [ 1.+0.j  1.+0.j  1.+0.j -1.+0.j -1.+0.j -1.+0.j]
  irrep (1, 1): chi = [ 1. +0.j    -0.5+0.866j -0.5-0.866j -1. +0.j     0.5-0.866j  0.5+0.866j]
  irrep (1, 2): chi = [ 1. +0.j    -0.5-0.866j -0.5+0.866j -1. +0.j     0.5+0.866j  0.5-0.866j]


## One-Dimensional Irreducible Representations of Finite Abelian Groups

### General Theory

A representation is a homomorphism $\rho:G\to GL(V)$. For a **finite abelian** group, every irrep is
**one-dimensional**. (i) Schur's lemma forces all $U_g$ in an irrep to be proportional to the
identity, so $\dim V=1$ and $\rho(g)\in U(1)$. (ii) The structure theorem decomposes
$G\cong\bigoplus_i \mathbb Z_{d_i}$, giving $|\hat G|=|G|$ irreps. (iii) For a cyclic factor
$\mathbb Z_n=\langle a\rangle$, the character is $\chi_k(a^m)=e^{2\pi i k m/n}$.

For translations on $L_1\times L_2$, group elements $\boldsymbol\delta=(\delta_1,\delta_2)$ and
irrep labels $\mathbf k=(k_1,k_2)$ range over the same set, with

\begin{equation}
\boxed{\chi_{\mathbf k}(\boldsymbol\delta) = \exp\Big[2\pi i\,(k_1\delta_1/L_1 + k_2\delta_2/L_2)\Big].}
\end{equation}

### Irrep Data Structure

`OneDim_Irrep(label, values)` stores the character values in the same order as
`Finite_Symmetry_Group.operations`. Irrep labels are `(k1, k2)` tuples. The **0-based** index into
`irrep_list` is what keys `ed_data.ed_scan_res` — a frequent source of confusion, so remember:
`ed_scan_res[irrep_idx]` uses 0-based `irrep_idx`, while the label is the 1-based momentum tuple.

> **Non-abelian caveat.** Combining abelian translations with abelian rotations is *not* abelian in
> general — they form a semidirect product, so 1D irreps do **not** resolve the full group and
> higher-dimensional projectors are required. That is beyond the scope of this package.


## Orbit–Stabilizer Decomposition

### Definitions

For a configuration $|\mathbf s\rangle$ (bitmask $m$):

- **Orbit**: $\mathrm{Orb}(\mathbf s) = \{U_g|\mathbf s\rangle : g\in G\}$.
- **Canonical representative**: the *smallest* bitmask in the orbit (lexicographic order):
  \begin{equation}\boxed{|[\mathbf s]\rangle = \arg\min_{|\mathbf s'\rangle\in\mathrm{Orb}(\mathbf s)} \mathrm{mask}(|\mathbf s'\rangle).}\end{equation}
  This is a gauge choice — any deterministic rule works; the smallest mask is convenient.
- **Stabilizer**: $\mathrm{Stab}(\mathbf s) = \{h\in G : U_h|[\mathbf s]\rangle = |[\mathbf s]\rangle\}$,
  with recorded stabilizer phases $U_h|[\mathbf s]\rangle = \alpha_h([\mathbf s])\,|[\mathbf s]\rangle$.

### Orbit–Stabilizer Theorem

\begin{equation}\boxed{|\mathrm{Orb}(\mathbf s)| = \frac{|G|}{|\mathrm{Stab}(\mathbf s)|}.}\end{equation}

**Proof.** Partition $G$ into left cosets of $\mathrm{Stab}(\mathbf s)$; each coset yields one
distinct state, so the number of states is the number of cosets $|G|/|\mathrm{Stab}(\mathbf s)|$.

**Implication.** Knowing only the $N_{\text{orbits}}$ representatives + stabilizer data reconstructs
the full sector, with compression ratio $N_{\text{orbits}}/\binom{N}{N_e}\approx 1/|G|$ — the
optimal $|G|$-fold reduction.


## Why Hash Tables Were the Bottleneck

Before the hash-free redesign, the orbit catalog and the canonicalization cache were built on two
hash tables. Documenting the *old* design makes the cost model of the new one concrete — and explains
why it had to go.

### The old construction

1. Enumerate all $\binom{N}{N_e}$ masks with Gosper's hack.
2. Maintain a `seen` set of already-visited masks (Python `set()`; Julia `Set{Mask}`). For each mask
   not in `seen`, compute its full orbit $\{U_g m\}_{g\in G}$ at $O(|G|)$ and insert every orbit
   member into `seen`.
3. Separately, maintain the canonicalization cache `CanonicalMap = Dict{Mask, Tuple{Mask, Int,
   ComplexF64}}` (Julia) / a Python `dict`, mapping each *scattered* mask to
   `(representative, g_idx, α)`, warmed lazily on first encounter.

Every `seen` insertion and every `Dict` probe hashes the mask (a single `int`/`UInt64`) and scans a
bucket; every `Dict` entry allocates a 3-tuple plus a key. A Julia `Dict` entry costs $\sim 64$+
bytes (key + value + hash-slot overhead), and a `Set` entry $\sim 50$–$100$ bytes per state once the
table is over-allocated for its load factor.

### The hot-loop probe

The matrix-free matvec and the H-block construction each performed, per scattered mask, one
canonicalization: hash the mask, scan the bucket, load the tuple `(repr, g_idx, α)`, then look `repr`
up in a **second** dict (the sector basis `repr_to_idx`). On a cache miss (mask not yet warmed) the
code fell back to the full $O(|G|)$ scan `get_canonical_representative`.

### Complexity

| Quantity | Old (hash-based) |
|---|---|
| Catalog construction | $O\!\big(\binom{N}{N_e}\,|G|\,k_{\mathrm{hash}}\big)$ time; $\sim 64$ B/state (`Dict`) + $\sim 50$–$100$ B/state (`Set`) |
| Canonical lookup | $O(1)$ amortized hash (high constant: hash + bucket scan + tuple allocation); $O(|G|)$ on miss |
| Matvec per-hop | one `Dict` probe (+ a second `Dict` probe for `repr → idx`) |

The constants are the problem: the hot loop thrashes pointer-chasing hash buckets and allocates
tuples, while the *memory* footprint of two hash tables over the full basis caps the reachable system
sizes. The redesign replaces both hash tables with dense, rank-indexed arrays — the subject of the
next three sections.


## Combinadic Rank: Hash-Free Basis Indexing

The first replacement is a **deterministic bijection** between the $k$-subsets of $\{0,\dots,N-1\}$
and the integer interval $[0,\binom{N}{k})$ — the **combinadic** (colexicographic) rank. It needs *no*
hash table and *no* per-state storage: a mask's rank is computed on the fly in $O(k)$ time from a
small shared binomial table. Throughout this section $k\equiv N_e$ (the code's `n_filled`).

### Colexicographic order

Write a $k$-subset as its sorted set-bit positions $b_0<b_1<\dots<b_{k-1}$. **Colex order** ranks two
subsets by their *largest* element first, then by the next-largest, and so on — equivalently the
reversed tuple $(b_{k-1},b_{k-2},\dots,b_0)$ is compared lexicographically. This is exactly the order
in which Gosper's hack enumerates fixed-popcount masks (smallest mask first), so rank $0$ is
$2^k-1=\texttt{0b}\underbrace{11\dots1}_{k}$.

### The hockey-stick tiling

To rank $S=\{b_0<\dots<b_{k-1}\}$, count the $k$-subsets that precede it in colex order, decomposing
by the largest element:

- subsets whose largest element is $<b_{k-1}$: exactly the $k$-subsets of $\{0,\dots,b_{k-1}-1\}$,
  of which there are $\binom{b_{k-1}}{k}$;
- among subsets with largest element $=b_{k-1}$, those whose second-largest element is $<b_{k-2}$:
  $\binom{b_{k-2}}{k-1}$ of them;
- $\vdots$
- finally, those agreeing with $S$ on the $k-1$ largest elements but with smallest element $<b_0$:
  $\binom{b_0}{1}$.

Hence

\begin{equation}
\boxed{\mathrm{rank}(m) = \sum_{j=0}^{k-1} \binom{b_j}{j+1}.}
\end{equation}

The **hockey-stick identity** $\sum_{b=j}^{n-1}\binom{b}{j}=\binom{n}{j+1}$ is what makes the
recursion collapse cleanly: the block of $k$-subsets with largest element *exactly* $b$ has size
$\binom{b}{k-1}$, and summing those blocks for $b<b_{k-1}$ gives $\binom{b_{k-1}}{k}$ — the first
rank term. The same identity guarantees the blocks tile $[0,\binom{N}{k})$ with no gaps, so the map is
a bijection.

### Worked example

For $N=4$, $k=2$, the six masks in colex order:

| mask (bin) | set bits $\{b_0,b_1\}$ | $\binom{b_0}{1}+\binom{b_1}{2}$ | rank |
|---|---|---|---|
| `0011` | $\{0,1\}$ | $0+0$ | 0 |
| `0101` | $\{0,2\}$ | $0+1$ | 1 |
| `0110` | $\{1,2\}$ | $1+1$ | 2 |
| `1001` | $\{0,3\}$ | $0+3$ | 3 |
| `1010` | $\{1,3\}$ | $1+3$ | 4 |
| `1100` | $\{2,3\}$ | $2+3$ | 5 |

In particular $\texttt{0b1010}$ (bits $\{1,3\}$) has rank $\binom{1}{1}+\binom{3}{2}=1+3=4$.

### Unranking (sketch)

To invert rank $r$: find the largest $b$ with $\binom{b}{k}\le r$; set $b_{k-1}=b$, subtract
$\binom{b}{k}$ from $r$, decrement $k$, and recurse. This is $O(Nk)$ with the binomial table. The
package does not ship it — Gosper's hack already enumerates masks in rank order — so it is omitted
here.

### Complexity and storage

- **Time**: $O(k)$ — one bit-isolate (`x & -x`), one `trailing_zeros`, and one table read per set bit.
- **Storage**: $O(1)$ per state (zero); the shared binomial table is $(N{+}1)\times(k{+}1)$ integers.

Julia ships this as `rank_combination(m)`; Python as `rank_combination(m, binom_table)` (with the
`binom_table` stored on the catalog).


In [3]:
from realspace_exactdiagonalization_py.symmetry_resolved_ed import rank_combination, _binom_table

n, k = 4, 2
binom_table = _binom_table(n, k)

masks = [0b0011, 0b0101, 0b0110, 0b1001, 0b1010, 0b1100]
print(f"colex rank table (N={n}, k={k}):")
for m in masks:
    bits = [i for i in range(n) if (m >> i) & 1]
    print(f"  {m:2d} = {m:04b}  bits={bits}  ->  rank = {rank_combination(m, binom_table)}")

print(f"\nrank(0b1010) = {rank_combination(0b1010, binom_table)}   (expected 4)")
ranks = sorted(rank_combination(m, binom_table) for m in masks)
print("bijection onto 0..C(4,2)-1:", ranks == list(range(6)))


colex rank table (N=4, k=2):
   3 = 0011  bits=[0, 1]  ->  rank = 0
   5 = 0101  bits=[0, 2]  ->  rank = 1
   6 = 0110  bits=[1, 2]  ->  rank = 2
   9 = 1001  bits=[0, 3]  ->  rank = 3
  10 = 1010  bits=[1, 3]  ->  rank = 4
  12 = 1100  bits=[2, 3]  ->  rank = 5

rank(0b1010) = 4   (expected 4)
bijection onto 0..C(4,2)-1: True


## LinTable: $O(1)$ Rank via Split Tables

The $O(k)$ combinadic rank is cheap, but when a rank sits on a lookup path it can be made **$O(1)$**
with a classic split-table scheme (Lin 1990; shipped by XDiag as `lin_table.hpp`). It trades a *tiny*
amount of memory for constant-time rank.

### Splitting the mask

Split the $N$-bit mask into an upper half $\mathrm{hi}$ (the $n_{\mathrm{left}}=N-\lfloor N/2\rfloor$
high bits) and a lower half $\mathrm{lo}$ (the $n_{\mathrm{right}}=\lfloor N/2\rfloor$ low bits).
Every fixed-filling mask is a pair $(\mathrm{hi},\mathrm{lo})$, and the particle number splits as
$k=\mathrm{popcount}(\mathrm{hi})+\mathrm{popcount}(\mathrm{lo})$.

### The two tables

1. **$L[u]$** — the number of $k$-subsets whose upper half is strictly less than $u$:

\begin{equation}
L[u] = \sum_{u'<u} \binom{n_{\mathrm{right}}}{k-\mathrm{popcount}(u')},
\end{equation}

because for a fixed upper half $u'$ the lower half may be any
$\big(k-\mathrm{popcount}(u')\big)$-subset of the $n_{\mathrm{right}}$ low bits.

2. **$R[v]$** — the colex rank of $v$ *within its own popcount class*, i.e. the plain combinadic rank
of the $n_{\mathrm{right}}$-bit mask $v$:

\begin{equation}
R[v] = \sum_j \binom{b_j}{j+1}\ \ \text{over the set bits of } v.
\end{equation}

Then

\begin{equation}
\boxed{\mathrm{rank}(m) = L[\mathrm{hi}] + R[\mathrm{lo}].}
\end{equation}

**Correctness.** Every $k$-subset with upper half $<\mathrm{hi}$ precedes $m$ in colex order and is
counted exactly once by $L[\mathrm{hi}]$. Among subsets with upper half $=\mathrm{hi}$, the lower half
ranges over all $\big(k-\mathrm{popcount}(\mathrm{hi})\big)$-subsets, ordered in colex, and
$R[\mathrm{lo}]$ is precisely $m$'s position inside that block. The two contributions are disjoint and
exhaustive.

### Worked example

$N=4$, $k=2$, $m=\texttt{0b1010}$. Then $n_{\mathrm{right}}=\lfloor4/2\rfloor=2$,
$n_{\mathrm{left}}=2$; $\mathrm{hi}=\texttt{0b10}=2$, $\mathrm{lo}=\texttt{0b10}=2$.

| $u$ | $L[u]=\sum_{u'<u}\binom{2}{2-\mathrm{pc}(u')}$ | $v$ | $R[v]$ |
|---|---|---|---|
| 0 | 0 | 0 (`00`) | 0 |
| 1 | $\binom{2}{2}=1$ | 1 (`01`) | 0 |
| 2 | $1+\binom{2}{1}=3$ | 2 (`10`) | $\binom{1}{1}=1$ |
| 3 | $3+\binom{2}{1}=5$ | 3 (`11`) | 0 |

Hence $\mathrm{rank}(\texttt{0b1010})=L[2]+R[2]=3+1=4$, matching the combinadic formula.

### Memory and the $N\le42$ gate

The tables hold $2^{n_{\mathrm{left}}}+2^{n_{\mathrm{right}}}\approx 2\cdot 2^{N/2}$ `Int64` entries.
At $N=42$ that is $2\cdot 2^{21}\approx 4.2\times10^6$ entries $\approx 33$ MB — XDiag's "ladder" cap.
For $N\le42$ the catalog builds and stores the tables (`lin_left`, `lin_right`, `lin_n_right`) and
`get_canonical` uses the $O(1)$ `lin_index`; for $N>42$ the tables are replaced by a sentinel
(`lin_n_right = -1`) and the code falls back to the $O(k)$ `rank_combination`.

### Julia / Python API

- Julia: `_lin_table(n_site, n_filled)` returns `(lin_left, lin_right, n_right)`; `lin_index(m,
  lin_left, lin_right, n_right)` is the $O(1)$ rank.
- Python: `_lin_table(n_site, n_filled)` and the Numba kernel `_lin_index_kernel(m, lin_left,
  lin_right, n_right)`, with the arrays stored on the catalog.


In [4]:
from math import comb
import numpy as np
import time as _t
from realspace_exactdiagonalization_py.symmetry_resolved_ed import (
    rank_combination, _lin_table, _lin_index_kernel, _binom_table)

# Build the tables for N=12, k=3
n_site, n_filled = 12, 3
lin_left, lin_right, lin_n_right = _lin_table(n_site, n_filled)
print("lin_left/lin_right shapes:", lin_left.shape, lin_right.shape, "  n_right =", lin_n_right)
print(f"memory: 2^{n_site - lin_n_right} + 2^{lin_n_right} = "
      f"{lin_left.size} + {lin_right.size} = {lin_left.size + lin_right.size} Int64 entries")

# Worked example: N=4, k=2, mask 0b1010 -> rank 4
ll, lr, nr = _lin_table(4, 2)
print("worked example: lin_index(0b1010) =",
      int(_lin_index_kernel(np.uint64(0b1010), ll, lr, np.int64(nr))), "  (expected 4)")

# Verify lin_index == rank_combination over the full C(12,3) = 220-mask basis
def gosper_next(x):
    c = x & -x
    r = x + c
    return (((r ^ x) >> 2) // c) | r

binom_table = _binom_table(n_site, n_filled)
x = (1 << n_filled) - 1
mismatch = 0
for _ in range(comb(n_site, n_filled)):
    if int(_lin_index_kernel(np.uint64(x), lin_left, lin_right, np.int64(lin_n_right))) != rank_combination(x, binom_table):
        mismatch += 1
    x = gosper_next(x)
print("lin_index vs rank_combination mismatches over all 220 masks:", mismatch)

# Timing on many masks: N=16, k=8 -> C(16,8) = 12870
ll16, lr16, nr16 = _lin_table(16, 8)
bt16 = _binom_table(16, 8)
x = (1 << 8) - 1
masks16 = []
for _ in range(comb(16, 8)):
    masks16.append(x)
    x = gosper_next(x)

def time_rank():
    return sum(rank_combination(m, bt16) for m in masks16)

def time_lin():
    return sum(int(_lin_index_kernel(np.uint64(m), ll16, lr16, np.int64(nr16))) for m in masks16)

time_rank(); time_lin()  # warm-up
t0 = _t.perf_counter(); time_rank(); t_rank = _t.perf_counter() - t0
t0 = _t.perf_counter(); time_lin();  t_lin  = _t.perf_counter() - t0
print(f"rank_combination (O(k), pure Python): {t_rank*1e3:.2f} ms over 12870 masks")
print(f"lin_index        (O(1), numpy):       {t_lin*1e3:.2f} ms over 12870 masks")


lin_left/lin_right shapes: (64,) (64,)   n_right = 6
memory: 2^6 + 2^6 = 64 + 64 = 128 Int64 entries
worked example: lin_index(0b1010) = 4   (expected 4)
lin_index vs rank_combination mismatches over all 220 masks: 0
rank_combination (O(k), pure Python): 20.63 ms over 12870 masks
lin_index        (O(1), numpy):       11.58 ms over 12870 masks


## Representative Tables: The Orbit Catalog Without Hashing

With the rank in hand, the orbit catalog is built **without any** `seen` set or `Dict` cache: the
entire fixed-filling basis is indexed by rank, and three dense arrays record, for every state, its
orbit and its canonicalization data.

### The three tables

Over the **full** fixed-filling basis (length $\binom{N}{N_e}$), indexed by the (LinTable / $O(k)$)
rank:

- `rep_rank_table[rank(m)]` — the **0-based** index of the orbit representative containing $m$
  ($-1$ = unset);
- `rep_sym_table[rank(m)]` — the **0-based** group-element index $g$ with $g(\mathrm{rep})=m$;
- `rep_amp_table[rank(m)]` — the canonicalization amplitude $\alpha_g(\mathrm{rep})$, stored as
  `complex64`.

### `isrepresentative`: the early-exit orbit-minimum test

`build_symmetry_orbit_catalog` runs Gosper's hack over all masks. For each mask $m$ it loops
$g\in G$; if **any** image satisfies $g(m)<m$ (as unsigned integers) then $m$ is not the orbit
minimum and is skipped immediately — no `seen` set, no hashing. The overwhelming majority of masks
are non-representatives, and for them the test aborts within a couple of probes; only the
$N_{\mathrm{orbits}}\approx\binom{N}{N_e}/|G|$ representatives run the full $|G|$ loop. (This is
XDiag's `isrepresentative`.)

### Fused orbit expansion (one pass, one write per state)

For each representative $\mathrm{rep}$, a **single** full pass over $G$ does two jobs at once: it
records the stabilizer (those $g$ with $g(\mathrm{rep})=\mathrm{rep}$, and their phases
$\alpha_g(\mathrm{rep})$) **and** writes every orbit member into the three tables via its rank.
Because orbits are disjoint, each full-basis state is written exactly once; the total cost is
$O(\binom{N}{N_e}\cdot|G|)$ with **no** hash-table constant.

### Last-write-wins is safe (abelian, sector-compatible orbits)

Within one representative's orbit, several group elements $g$ map $\mathrm{rep}$ to the *same* state
$m$ — precisely the coset $g\,\mathrm{Stab}(\mathrm{rep})$. The fused loop writes each such $g$ in
turn, so the last one wins. This is harmless because the projection coefficient
$\mathsf{coeff}=\alpha_g(\mathrm{rep})\,\chi(g)^*$ is *independent* of the coset representative: if
$g_2=g_1 h$ with $h\in\mathrm{Stab}(\mathrm{rep})$, then

\begin{align}
\alpha_{g_2}(\mathrm{rep}) &= \alpha_{g_1}(\mathrm{rep})\,\alpha_h(\mathrm{rep}), &
\chi(g_2)^* &= \chi(g_1)^*\,\chi(h)^*,
\end{align}

where the first identity follows from $U_{g_1 h}=U_{g_1}U_h$ and
$U_h|\mathrm{rep}\rangle=\alpha_h(\mathrm{rep})|\mathrm{rep}\rangle$. For an orbit that belongs to the
sector (the irrep-compatibility condition of §"Irrep-Induced Projector and the Symmetry-Resolved
Basis"), $\alpha_h(\mathrm{rep})=\chi(h)$ for every $h\in\mathrm{Stab}(\mathrm{rep})$, hence
$\alpha_h(\mathrm{rep})\,\chi(h)^*=\alpha_h(\mathrm{rep})\,\alpha_h(\mathrm{rep})^*=1$, and

\begin{equation}
\alpha_{g_2}(\mathrm{rep})\,\chi(g_2)^* = \alpha_{g_1}(\mathrm{rep})\,\chi(g_1)^* .
\end{equation}

So the amplitude that survives the last write yields the correct projection coefficient, regardless
of which coset representative happened to be stored.

### The $U(1)$-phase generalization (why we store the amplitude)

XDiag's symmetries are *pure permutations* (translations/rotations with $\eta_g(i)=1$), so its
representative table stores only the group element $g$ plus a one-bit Jordan–Wigner sign and applies
$\chi(g)$ on the fly. Our symmetry action carries **per-site** $U(1)$ phases $\eta_g(i)$
(flux-threaded translations), so

\begin{equation}
\alpha_g(\mathrm{rep}) = \prod_{i\in\mathrm{occ}(\mathrm{rep})}\eta_g(i)
\end{equation}

depends on *which* sites are occupied — it cannot be reconstructed from $g$ and $\chi(g)$ alone. We
therefore store the full complex amplitude in `rep_amp_table`. This is the one substantive
generalization over XDiag's tables.

### Memory: 16 B/state vs $\sim 64$+ B/state

The three tables cost $4+4+8=16$ bytes per basis state (`int32` + `int32` + `complex64`), versus
$\sim 64$+ bytes per entry for the old `Dict` cache (plus the separate $\sim 50$–$100$ B/state `Set`),
with no hash-slot overhead and no pointer chasing. The LinTable adds only $2\cdot 2^{N/2}\cdot 8$
bytes *in total* (shared across all states).


In [5]:
from fractions import Fraction
import realspace_exactdiagonalization_py as ed

model = ed.build_zero_flux_bosonic_fci_second_quantized_model(sample_size=[2, 3], params=ed.params_DNSheng)
G = ed.build_translation_group(model.lattice)
ed_data = ed.build_ed_data(model, filling_fraction=Fraction(1, 4), symmetry_group=G)
cat = ed_data.orbit_catalog

n = len(cat.rep_rank_table)
print("n_orbits =", len(cat.representative_mask_list), "| full basis C(12,3) =", n)
print("rep_rank_table:", cat.rep_rank_table.dtype, cat.rep_rank_table.shape,
      "| unset(-1) entries =", int((cat.rep_rank_table == -1).sum()))
print("rep_sym_table :", cat.rep_sym_table.dtype, cat.rep_sym_table.shape)
print("rep_amp_table :", cat.rep_amp_table.dtype, cat.rep_amp_table.shape)
print("lin_left/lin_right:", cat.lin_left.shape, cat.lin_right.shape, "| lin_n_right =", cat.lin_n_right)

mem = 4 * n + 4 * n + 8 * n
print(f"\nrepresentative tables: {mem} B total = {mem/n:.1f} B/state;  "
      f"LinTable: {8*cat.lin_left.size + 8*cat.lin_right.size} B")

cmap = ed.CanonicalMap(G, model.particle_statistics, cat)
m = cat.representative_mask_list[3]
print("\nO(1) canonical lookup: get_canonical(rep[3] =", m, ") =", ed.get_canonical(cmap, m))


	Building symmetry-orbit catalog (n_filled=3, |G|=6) ... Done. 38 orbits (reduction 17.3%).  t=3.007s
n_orbits = 38 | full basis C(12,3) = 220
rep_rank_table: int32 (220,) | unset(-1) entries = 0
rep_sym_table : int32 (220,)
rep_amp_table : complex64 (220,)
lin_left/lin_right: (64,) (64,) | lin_n_right = 6

representative tables: 3520 B total = 16.0 B/state;  LinTable: 1024 B

O(1) canonical lookup: get_canonical(rep[3] = 21 ) = (21, 0, (1+0j))


## Fast Bitwise Algorithms

### Gosper's Hack

To enumerate all masks with exactly $k$ set bits, iterate directly over the fixed-Hamming-weight
integers in lexicographic order — $O(1)$ per step, zero allocation:

```python
def gosper_next(x):
    c = x & -x           # rightmost 1-bit
    r = x + c            # carry chain
    return (((r ^ x) >> 2) // c) | r
```

The first mask is $2^k-1$; iteration stops past $2^N$. The demo cell above already ran it.

### Fast Symmetry Action on a Bitmask

`apply_operation_to_mask` processes *only the occupied sites* ($O(N_e)$) by repeatedly isolating the
rightmost set bit (`x & -x`), mapping it to its destination, and clearing it (`x ^= lsb`), while
accumulating the phase. The fermionic variant additionally tracks the permutation parity with a
`popcount` check per move.


In [6]:
import realspace_exactdiagonalization_py as ed

# A transposition (odd permutation): swap sites 1 and 2 of a 3-site system
op = ed.Symmetry_Operation("swap(1,2)", [2, 1, 3])

m = 0b111  # all three sites occupied
new_b, phase_b = ed.apply_operation_to_mask(m, op, ed.Particle_Statistics.BOSONIC)
new_f, phase_f = ed.apply_operation_to_mask(m, op, ed.Particle_Statistics.FERMIONIC)
print(f"boson  : m -> {bin(new_b)}, phase = {phase_b:+}")
print(f"fermion: m -> {bin(new_f)}, phase = {phase_f:+}")
# Bosons see the pure permutation (+1); fermions pick up the sign of the
# permutation restricted to occupied sites (-1 for an odd transposition).


boson  : m -> 0b111, phase = (+1+0j)
fermion: m -> 0b111, phase = (-1+0j)


## Irrep-Induced Projector and the Symmetry-Resolved Basis

### Construction of the Projector

For a 1D irrep $\chi$,

\begin{equation}\boxed{P_\chi := \frac{1}{|G|}\sum_{g\in G}\chi(g)^*\, U_g .}\end{equation}

Using $\chi(g)\chi(h)=\chi(gh)$ and $U_g U_h=U_{gh}$, one shows $P_\chi^2=P_\chi$; since
$[U_g,H]=0$ and $P_\chi^\dagger=P_\chi$, the projector block-diagonalizes $H$.

### Projection of an Orbit Representative

Splitting the sum over $G$ into cosets of $\mathrm{Stab}(\mathbf s)$:

\begin{align}
P_\chi|[\mathbf s]\rangle
&= \frac{1}{|G|}\sum_{g\in G/\mathrm{Stab}(\mathbf s)}\chi(g)^*\,U_g|[\mathbf s]\rangle
   \underbrace{\Big(\sum_{h\in\mathrm{Stab}(\mathbf s)} \chi(h)^*\,\alpha_h([\mathbf s])\Big)}_{S}.
\end{align}

By character orthogonality on the subgroup, $S=|\mathrm{Stab}|$ iff
$\chi(h)=\alpha_h([\mathbf s])\ \forall h\in\mathrm{Stab}(\mathbf s)$, and $S=0$ otherwise. This
gives the **irrep compatibility condition**:

\begin{equation}
\boxed{\chi(h)=\alpha_h([\mathbf s])\ \ \forall h\in\mathrm{Stab}(\mathbf s)
\;\;\Longleftrightarrow\;\; \text{orbit }[\mathbf s]\text{ contributes to irrep }\chi .}
\end{equation}

### Orthonormalization

The projected state is not normalized; $|P_\chi|[\mathbf s]\rangle|^2 = |\mathrm{Stab}|/|G|$.
Hence the **orthonormal projected basis state** is

\begin{equation}
\boxed{|\widetilde{[\mathbf s];\chi}\rangle := \sqrt{\frac{|G|}{|\mathrm{Stab}(\mathbf s)|}}\,P_\chi|[\mathbf s]\rangle
= \sqrt{\frac{|\mathrm{Stab}(\mathbf s)|}{|G|}}\sum_{g\in G/\mathrm{Stab}(\mathbf s)} \chi(g)^*\,U_g|[\mathbf s]\rangle .}
\end{equation}

The `Symmetry_Sector_Basis` stores exactly the compatible representatives and their
$|\mathrm{Stab}|$, plus a `dict` mapping representative $\to$ 0-based sector index.


## Irrep Projection of an Arbitrary Configuration

When $H$ acts on a representative $|[\mathbf s]\rangle$ it produces a **scattered** configuration
$|\mathbf m\rangle = H|[\mathbf s]\rangle$ that is generally *not* a representative. To compute
matrix elements we project it back with `project_to_sector` (or the internal
`get_canonical`/`get_canonical_representative`):

1. Find the canonical representative $|[\mathbf m]\rangle$ and the group element $g_{\mathbf m}$
   with $U_{g_{\mathbf m}}|\mathbf m\rangle=\alpha_{g_{\mathbf m}}|[\mathbf m]\rangle$.
2. Then $P_\chi|\mathbf m\rangle = \alpha_{g_{\mathbf m}}\chi(g_{\mathbf m})^*\, P_\chi|[\mathbf m]\rangle$.
3. In the orthonormal basis, $P_\chi|\mathbf m\rangle = \mathsf{coeff}\,\sqrt{|\mathrm{Stab}(\mathbf m)|/|G|}\,|\widetilde{[\mathbf m];\chi}\rangle$ with
   $\mathsf{coeff}=\alpha_{g_{\mathbf m}}\chi(g_{\mathbf m})^*$.

The returned `(row_idx, coeff)` directly gives the non-zero entries of the Hamiltonian matrix in the
orthonormal projected basis. The `CanonicalMap` is a **thin view** over the orbit catalog's
representative tables: `get_canonical(m)` computes the LinTable $O(1)$ rank (or the $O(k)$ combinadic fallback for $N>42$) and does three array reads
(`rep_rank_table`/`rep_sym_table`/`rep_amp_table`) — $O(1)$ with **no hashing**. `populate_canonical_map`
is a no-op (the tables are prebuilt at catalog time; kept for API parity). The tables are read to
**build** the CSR block (matrix mode) or the projection table (matrix-free mode).


## Hamiltonian Matrix Construction in the Symmetry Sector

### Matrix Elements

The matrix element in the orthonormal projected basis is

\begin{align}
H_{\mathbf s',\mathbf s}^{\chi}
&= \langle\widetilde{[\mathbf s'];\chi}|H|\widetilde{[\mathbf s];\chi}\rangle \nonumber\\
&= \sqrt{\frac{|G|}{|\mathrm{Stab}(\mathbf s)|}}\,\langle\widetilde{[\mathbf s'];\chi}|\,P_\chi\,|\mathbf m\rangle \nonumber\\
&= \mathsf{coeff}\cdot\sqrt{\frac{|\mathrm{Stab}(\mathbf m)|}{|\mathrm{Stab}(\mathbf s)|}}\,\delta_{[\mathbf s'],[\mathbf m]},
\qquad \mathsf{coeff}=\alpha_{g_{\mathbf m}}\chi(g_{\mathbf m})^* .
\end{align}

**Key result:** each matrix element is a raw scattering amplitude (the hopping $t$, a Jordan–Wigner
phase if fermionic, and $\mathsf{coeff}$) times a rescaling
$\sqrt{|\mathrm{Stab}(\mathbf m)|/|\mathrm{Stab}(\mathbf s)|}$ that corrects for different
stabilizer sizes of initial and final representatives. For diagonal density terms $|\mathbf{m}\rangle=V n_i n_j |[\mathbf{s}]\rangle$ is the representative itself, so $\mathsf{coeff}=1$ and the
$\sqrt{\cdot}$ factor is $1$.

### Sparse Matrix Construction (matrix mode)

`build_ed_Hamiltonian_symmetry_block(basis, bilinear_terms, density_terms, cmap)` loops over
representatives (columns), accumulates the diagonal density terms, then for each valid hop builds
the scattered mask, projects it (`get_canonical`), and appends the COO entry
$t\cdot\mathrm{phase}\cdot\mathsf{coeff}\cdot\sqrt{|\mathrm{Stab}_{\mathrm{row}}|/|\mathrm{Stab}_{\mathrm{col}}|}$.
The result is a `scipy.sparse.csr_matrix` (the scipy analogue of the Julia `SparseMatrixCSC`).


In [7]:
import numpy as np
from fractions import Fraction
import realspace_exactdiagonalization_py as ed

model = ed.build_zero_flux_bosonic_fci_second_quantized_model(
    sample_size=[2, 3], params=ed.params_DNSheng)
G = ed.build_translation_group(model.lattice)
ed_data = ed.build_ed_data(model, filling_fraction=Fraction(1, 4), symmetry_group=G)

irrep = ed_data.irrep_list[0]          # k = (0,0)
basis = ed.build_symmetry_sector_basis(ed_data.orbit_catalog, irrep)
cmap = ed.CanonicalMap(ed_data.symmetry_group, model.particle_statistics,
                       ed_data.orbit_catalog)
H = ed.build_ed_Hamiltonian_symmetry_block(
    basis, model.bilinear_terms, model.density_density_terms, cmap)
print("sector dim =", len(basis.representative_mask_list), "| nnz =", H.nnz)

vals_dense, vecs_dense = ed.diagonalize_block_dense(H, nev=3)
vals_arpack, vecs_arpack = ed.diagonalize_block_arpack(H, nev=3)
print("dense  :", vals_dense)
print("arpack :", vals_arpack)
assert np.allclose(vals_dense, vals_arpack, atol=1e-9), "dense vs arpack mismatch"
print("✓ dense and ARPACK agree (sector dim < 500 → both exact); "
      "ground state E0(k=(0,0)) =", f"{vals_dense[0]:.15f}")


	Building symmetry-orbit catalog (n_filled=3, |G|=6) ... Done. 38 orbits (reduction 17.3%).  t=0.0s
	Building H block (matrix mode) @ irrep (0, 0) (dim=38) ... Done. nnz=756, sparsity=0.523546. t=2.887s
sector dim = 38 | nnz = 756
dense  : [-7.16380536 -6.62646328 -6.40131123]
arpack : [-7.16380536 -6.62646328 -6.40131123]
✓ dense and ARPACK agree (sector dim < 500 → both exact); ground state E0(k=(0,0)) = -7.163805363423434


## The Hot Loop: Projection Table + Branch-Free Hopping

The final piece removes canonicalization from the matvec entirely. Two modes remain: **matrix** mode
builds an explicit sparse CSR block (fast, memory-heavy); **matrix-free** mode stores only a flat
projection table and applies $H$ as a gather–scatter kernel (near-zero memory).

### The projection table

`build_matrixfree_projection_table(basis, bilinear_terms, density_terms, cmap)` runs **once per
sector** and precomputes, for every valid hopping move, the flat triplet

\begin{equation}
(\mathrm{row},\ \mathrm{column},\ \mathrm{amplitude}),\qquad
\mathrm{amplitude} = t\cdot s_{\mathrm{JW}}\cdot \alpha_g(\mathrm{rep})\cdot \chi(g)^*
\cdot\sqrt{\frac{|\mathrm{Stab}_{\mathrm{row}}|}{|\mathrm{Stab}_{\mathrm{col}}|}},
\end{equation}

plus the per-column diagonal (density–density) term. This is exactly the H-block matrix element of
§"Hamiltonian Matrix Construction in the Symmetry Sector", frozen into arrays.

### The gather–scatter kernel

`MatrixFreeHamiltonian.__call__(x)` is then a pure gather–scatter kernel: for each edge $e$ it reads
`x[col_ind[e]]`, multiplies by `table_vals[e]`, and scatters into a per-thread
`y[thread][row_ind[e]]` buffer, with a final reduction over threads. **Zero** rank computations,
**zero** dict probes, **zero** $O(|G|)$ scans per hop. This replaced the old design in which every
scattered mask was re-projected through the `CanonicalMap` `Dict` on every matvec.

### Branch-free XOR hop (XDiag trick 3a)

For a hop $i\to j$ define `hop_mask = bitmask(i) | bitmask(j)` and `from_mask = bitmask(i)`. A single
fused gate

\begin{equation}
(m \;\&\; \mathrm{hop\_mask}) \;==\; \mathrm{from\_mask}
\end{equation}

tests *both* occupancy conditions at once — site $i$ occupied and site $j$ empty — because the two
bits of `hop_mask` must read exactly $(1,0)$. When it passes, the hop is applied with one branch-free
XOR,

\begin{equation}
m \;\to\; m \;\oplus\; \mathrm{hop\_mask},
\end{equation}

flipping both bits simultaneously (no separate clear-then-set). The masks are precomputed once per
term list.

### Per-hop cost, old vs new

| Step | Old (Dict) | New (projection table) |
|---|---|---|
| valid-hop test | two occupancy branches | one AND + equality (`trick 3a`) |
| state update | clear + set (two ops) | one XOR |
| canonicalization | hash + bucket scan + tuple load (+ $O(|G|)$ on miss) | none (precomputed) |
| sector lookup | second `Dict` probe | none |
| write | — | one scatter into a thread buffer |

The precomputation is the *same* work as one H-block construction, so matrix-free mode pays
$O(\mathrm{nnz})$ once and $O(\mathrm{nnz})$ per matvec with tiny constants.

### Measured numbers

On the Haldane $[2,7]$ sector (dimension $84\,576$), the matrix-free matvec dropped from
**563.8 ms → 31.3 ms** after switching to the projection table, with bit-identical energies. Full
single-sector timings (matrix vs matrix-free) are collected in the complexity summary below.


In [8]:
from fractions import Fraction
import numpy as np
import realspace_exactdiagonalization_py as ed

model = ed.build_zero_flux_bosonic_fci_second_quantized_model(sample_size=[2, 3], params=ed.params_DNSheng)
G = ed.build_translation_group(model.lattice)
ed_data = ed.build_ed_data(model, filling_fraction=Fraction(1, 4), symmetry_group=G)
irrep = ed_data.irrep_list[0]
basis = ed.build_symmetry_sector_basis(ed_data.orbit_catalog, irrep)
cmap = ed.CanonicalMap(G, model.particle_statistics, ed_data.orbit_catalog)

H = ed.build_ed_Hamiltonian_symmetry_block(basis, model.bilinear_terms, model.density_density_terms, cmap)
row_ind, col_ind, vals, h_diag = ed.build_matrixfree_projection_table(
    basis, model.bilinear_terms, model.density_density_terms, cmap)
print("sector dim =", len(basis.representative_mask_list), "| nnz(H) =", H.nnz,
      "| projection table entries =", len(row_ind))

H_mf = ed.MatrixFreeHamiltonian(basis, model.bilinear_terms, model.density_density_terms, cmap)
rng = np.random.default_rng(0)
x = rng.standard_normal(len(basis.representative_mask_list)) + 1j * rng.standard_normal(len(basis.representative_mask_list))
y_matrix = H @ x
y_mf = H_mf(x)
print("max |H·x - MatrixFreeH(x)| =", np.max(np.abs(y_matrix - y_mf)))


	Building symmetry-orbit catalog (n_filled=3, |G|=6) ... Done. 38 orbits (reduction 17.3%).  t=0.0s
	Building H block (matrix mode) @ irrep (0, 0) (dim=38) ... Done. nnz=756, sparsity=0.523546. t=0.0s
sector dim = 38 | nnz(H) = 756 | projection table entries = 1120
max |H·x - MatrixFreeH(x)| = 3.552713678800501e-15


## Fermionic Particle Statistics: Jordan–Wigner Strings

Fermions require two sign structures. (i) **Permutation parity** in `apply_operation_to_mask`:
placing particles in canonical order, each move to site $p$ accumulates a $(-1)$ when an odd number
of already-placed particles sit above $p$ — tracked with
$\mathrm{popcount}(\mathrm{new\_mask} \gg p)$.

(ii) **Jordan–Wigner string** in `hopping_phase_for_stats`: a hop $i\to j$ in configuration $m$
carries

\begin{equation}
(-1)^{\#\{\text{occupied }k:\ \min(i,j)<k<\max(i,j)\}},
\end{equation}

i.e. the parity of occupied sites *between* $i$ and $j$ in the flattened (1D) site ordering. The
flattened-graph ordering must be chosen so that fermion anticommutation is reproduced correctly;
the spinful-Hubbard builder uses interleaved $(\uparrow_1,\downarrow_1,\uparrow_2,\downarrow_2,\ldots)$
for exactly this reason.


## Flux-Aware Symmetry and Twisted Boundary Conditions

Threading a flux $\boldsymbol\theta$ (in units of $2\pi$) through the periodic directions multiplies
boundary-crossing hoppings by a Peierls phase. The Hamiltonian $H(\boldsymbol\theta)$ no longer
commutes with the bare translation $T^0$, but it **does** commute with the gauge-covariant
translation

\begin{equation}
T^{\boldsymbol\theta}_{(d_x,d_y)} = G_{\boldsymbol\theta}\, T^0_{(d_x,d_y)}\, G_{\boldsymbol\theta}^{-1},
\qquad
G_{\boldsymbol\theta} = \exp\Big(i2\pi\sum_j \frac{\boldsymbol\theta\cdot x_j}{L}\,\hat n_j\Big).
\end{equation}

`build_translation_group(lattice, θ)` absorbs $G_{\boldsymbol\theta}$ into the per-site
`perm_phases` of each `Symmetry_Operation`, so the **irrep labels (momenta) stay fixed** under flux.
The orbit *partition* is unchanged (it depends only on the permutation part), so only the stabilizer
phases need recomputation — `update_orbit_stabilizer_phases` does this in $O(N_{\text{orbits}}\cdot|G|)$
instead of re-running Gosper's hack.


In [9]:
import math
import realspace_exactdiagonalization_py as ed
from tightbinding_py import initialize_real_space_lattice

lattice = initialize_real_space_lattice(
    sample_size=[2, 2],
    brav_vec_list=[[1.0, 0.0], [1 / 2, math.sqrt(3) / 2]],
    sub_crys_list=[[0.0, 0.0], [1 / 3, 1 / 3]],
    lattice_name="Haldane_Honeycomb",
    pbc_indicator=[True, True],
)
G0 = ed.build_translation_group(lattice)                # zero flux
Gt = ed.build_translation_group(lattice, [0.25, 0.0])   # θ_x = 2π × 0.25
for name, G in [("θ = 0      ", G0), ("θ_x = 0.25·2π", Gt)]:
    op = G.operations[2]  # translation by (1,0) — crosses the x-boundary
    print(f"{name}: label {op.label}, perm_phases = {op.perm_phases.round(3)}")
print("irrep labels unchanged:",
      [ir.label for ir in ed.build_irrep_list(G0, lattice)],
      "==", [ir.label for ir in ed.build_irrep_list(Gt, lattice)])


θ = 0      : label (1, 0), perm_phases = [1.+0.j 1.+0.j 1.+0.j 1.+0.j 1.+0.j 1.+0.j 1.+0.j 1.+0.j]
θ_x = 0.25·2π: label (1, 0), perm_phases = [0.707-0.707j 0.707-0.707j 0.707+0.707j 0.707+0.707j 0.707-0.707j
 0.707-0.707j 0.707+0.707j 0.707+0.707j]
irrep labels unchanged: [(0, 0), (0, 1), (1, 0), (1, 1)] == [(0, 0), (0, 1), (1, 0), (1, 1)]


## Checkpointing

`save_checkpoint(ed_data, path)` pickles the full `Symmetry_Resolved_ED_Data` atomically (a `.tmp`
sibling + `os.replace`), and `load_checkpoint(path)` restores it. In flux-scan mode, `ed_scan`
writes one canonical per-θ file per flux point via `ed_scan_checkpoint_filename` (which embeds the
model name, sample size, filling, twisted phases and rounded params), so spectrum-flow and
charge-pump observables can cheaply **resume** without recomputing completed sectors. (The Julia
package uses JLD2; the Python port uses `pickle` — see the differences section.)


## Implementation Differences from the Julia Engine (with Reasons)The port is *numerically faithful* (energies agree to $\lesssim 2\times10^{-13}$) but re-maps eachJulia abstraction onto the idiomatic Python/NumPy/SciPy stack — **pure numpy, no Numba** (see the"A/B: Numba vs pure numpy" section below for the benchmark that motivated the choice). After the XDiag study, the twopackages **converged on a shared core design** — the XDiag-style dense representative tables(`rep_rank_table`/`rep_sym_table`/`rep_amp_table`, combinadic rank + `isrepresentative`) and thematrix-free projection table are now identical in both languages — so those are *not* listed asdifferences below. What remains:| Julia | Python | Reason ||---|---|---|| `Threads.@threads :static` | vectorized numpy (byte-lookup group actions, two-pass rep test, per-term gather/scatter, single CSR matvec) | No Julia-style task scheduler in Python; chunked numpy vectorization is faster than the former Numba `prange` layer (see the A/B section) || `SparseMatrixCSC` + `Arpack.jl` | `scipy.sparse.csr_matrix` + `scipy.sparse.linalg.eigs` | SciPy ships ARPACK; CSR is its native format (both use the same `which=:SR`/`"SR"` smallest-real-part convention — see below) || `KrylovKit.eigsolve` (matrix-free) | `scipy.sparse.linalg.LinearOperator` + `eigs` | Avoid an extra dependency; SciPy's ARPACK already accepts `LinearOperator` || `JLD2` checkpoints | `pickle` | No JLD2 in Python; pickle is stdlib and round-trips the dataclasses || `MLStyle.@data Bosonic()/Fermionic()` | `enum.Enum` `Particle_Statistics.BOSONIC/FERMIONIC` | Python has no algebraic-data-type dispatch; a flag-checked enum is the closest idiom || `Distributed.pmap` (multi-process) | *absent* — shared-memory only | **Both languages are multithreading-only** (the Julia `Distributed` layer was removed too: it re-serialized the ~100 MB-scale representative tables to every worker per sector, causing per-process memory blow-ups at large sizes; pure `Threads.@threads` / vectorized numpy is faster and memory-bounded) || `UInt64` masks | Python `int` masks (public API) + `np.uint64` inside kernels | Python ints are arbitrary-precision and idiomatic; kernels downcast to `uint64` for speed || 1-based dict keys (Julia) | **0-based** `ed_scan_res` keys & irrep indices | Python lists/dicts are 0-based; documented explicitly |Two consequences worth internalizing: (1) `ed_data.ed_scan_res[irrep_idx]` is keyed by the **0-based**index into `irrep_list`, not by the `(k1,k2)` label; (2) `identity_idx=0` (not 1) and all`stabilizer_g_indices_list` entries are 0-based. Site indices in model terms remain **1-based** inboth languages.### Diagonalization convention (non-Hermitian gauge blocks)A subtle but important **shared** convention (not a difference): for flux-aware symmetry groups theprojected sector block can be *non-Hermitian in this gauge* — the gauge-covariant translations canmake some self-loop (diagonal) hopping phases complex. Both languages therefore diagonalize the**raw** block identically and never symmetrize it:| Case | Julia | Python ||---|---|---|| $n\le500$ (matrix or matrix-free) | `eigen(Hermitian(Matrix(H)))` — LAPACK `zheev` on the **upper** triangle | `np.linalg.eigh(H_dense, UPLO="U")` || $n>500$, matrix mode | `Arpack.eigs(H; which=:SR)` — *general* (non-Hermitian) smallest-real-part | `scipy.sparse.linalg.eigs(H, which="SR")` || $n>500$, matrix-free | `KrylovKit.eigsolve(H_op, x0, nev, :SR)` on the raw operator | `scipy.sparse.linalg.eigs(LinearOperator, which="SR")` |In particular the port does **not** call `eigsh` and does **not** symmetrize `(H+H^\dagger)/2`: thesmallest-*real-part* (`which="SR"`) solves act on the raw block and the eigenvalues are sortedascending. This matches the Julia reference to $\max|\Delta E| = 10^{-12}$ over a $7\times7$ fluxgrid $\times$ 2 sectors.

### A/B: Numba vs pure numpy (why the Numba layer was removed)

The first Python port used Numba `njit`/`prange` kernels (mirroring the Julia hot loops).
A controlled A/B on the Heisenberg chain $N=24$ (same inputs, bit-identical outputs)
showed that **chunked numpy vectorization beats Numba on every hot path**:

| Kernel | Numba | Pure numpy (vectorized) | ratio |
|---|---|---|---|
| Orbit-catalog build | 1.25 s | 1.21 s | **0.97×** |
| H-block construction | 0.23 s | 0.13 s | **0.56×** |
| Projection table | 0.23 s | 0.12 s | **0.53×** |
| One matvec | 4.15 ms | 2.61 ms (CSR) | **0.63×** |

The numpy versions are faithful ports of the *same* algorithms, but vectorized over chunks
of the fixed-filling basis:

- **Group action = byte-lookup gathers.**  For every group element $g$ and byte position $b$,
  a 256-entry table maps an input byte to its permuted `uint64` image (and to the accumulated
  U(1) angle).  Applying $g$ to a chunk is then `n_bytes` C-level `np.take` calls instead of
  per-(g, bit) scalar loops.
- **Two-pass representative test.**  Pass 1 min-reduces the images over $g$ (the
  `isrepresentative` early-exit, vectorized); pass 2 recomputes images/phases for the found
  representatives only (≈ $B/|G|$ of the chunk) and scatters into the dense tables.
- **Colex unranking.**  Gosper's sequential walk is replaced by a vectorized combinadic
  unranking per chunk (`searchsorted` over the binomial table).
- **Single CSR matvec.**  The matrix-free projection table *is* the sector Hamiltonian in
  coordinate form; converting once to CSR makes each $H|x\rangle$ one sparse matvec
  (BLAS-level gather) — faster than the Numba per-thread scatter.

Removing Numba also removes the JIT warmup cost and the `NUMBA_NUM_THREADS` configuration
surface: the runtime stack is numpy + scipy only.


### XDiag study — adopted vs deferredAdopted in both languages: combinadic-rank representative tables (no hashing), the LinTable $O(1)$rank (gated to $N\le42$), the matrix-free projection table (gather–scatter matvec), and branch-freeXOR hopping with a direction-aware mask gate.  Deferred with reasons: bit-packed per-state arrays(the plain tables are already $4$–$10\times$ smaller than the old caches; bit-packing is future workfor very large systems), parallel two-phase COO build (the shared-memory construction is alreadyfast with `Threads.@threads`; an earlier `Distributed.pmap` layer was removed), NonBranchingOp tables (all current models are two-siteterms), Simon reorthogonalization / LOBPCG (KrylovKit/ARPACK suffice).

## Complexity Summary and Why This Matters

### Old vs new

| Quantity | Old (hash-based) | New (hash-free) |
|---|---|---|
| Catalog construction | $O\!\big(\binom{N}{N_e}\,|G|\,k_{\mathrm{hash}}\big)$; $\sim 64$ B/state `Dict` + $\sim 50$–$100$ B/state `Set` | $O\!\big(\binom{N}{N_e}\,|G|\big)$; $16$ B/state dense tables |
| Canonical lookup | $O(1)$ hash amortized (high constant) + $O(|G|)$ fallback | $O(1)$ two array reads (LinTable, $N\le42$) or $O(k)$ rank ($N>42$); no fallback |
| Matvec per-hop | `Dict` probe (+ second `Dict` probe) | array gather (projection table) |
| Per-state memory | $\sim 114$–$164$ B (Set + Dict) | $16$ B (rep tables) + $2\cdot 2^{N/2}\cdot 8$ B shared LinTable |

### Single-sector benchmark (latest)

H-block construction for the Heisenberg chain at $N=24$ is $\sim 0.3$ s (Julia). Single-sector times
(seconds):

| Model | Julia matrix | Julia matrixfree | Python matrix | Python matrixfree |
|---|---|---|---|---|
| Heisenberg $N=24$ | 1.8564 | 1.9603 | 1.8551 | 3.1590 |
| Haldane $3\times4$ | 0.5502 | 0.6251 | 0.4678 | 1.0809 |
| Hubbard $2\times5$ | 0.4592 | 0.5508 | 0.2967 | 0.6614 |

Energies agree between the two languages (and between matrix and matrix-free modes) to
$\le 10^{-13}$.

### Why this matters

The design is a direct descendant of XDiag's lookup machinery: combinadic rank for the basis index,
the LinTable for constant-time rank below $N=42$, and dense representative tables built by
`isrepresentative` + orbit expansion. The one substantive generalization is the **$U(1)$-phase
amplitude**: because our symmetry action carries per-site flux phases, `rep_amp_table` stores the full
state-dependent $\alpha_g(\mathrm{rep})$ instead of XDiag's group-element-plus-JW-sign. Deferred XDiag
tricks — bit-packed per-state arrays, the parallel two-phase COO build, and `NonBranchingOp` tables —
are future work; bit-packing in particular is now straightforward because the tables are already dense
rank-indexed arrays.


## Key Formulas — Quick Reference

| Quantity | Formula |
|---|---|
| Bitmask | $m=\sum_i n_i 2^{i-1}$ |
| Combinadic rank | $\mathrm{rank}(m)=\sum_j \binom{b_j}{j+1}$ over set bits $b_0<b_1<\dots$ |
| LinTable rank | $\mathrm{rank}(m)=L[\mathrm{hi}]+R[\mathrm{lo}]$ |
| Symmetry action | $U_g a_i^\dagger U_g^{-1}=\eta_g(i)a_{\pi_g(i)}^\dagger$ |
| Orbit–stabilizer | $|\mathrm{Orb}(\mathbf s)|=|G|/|\mathrm{Stab}(\mathbf s)|$ |
| Projector | $P_\chi=\frac{1}{|G|}\sum_g \chi(g)^* U_g$ |
| Compatibility | $\chi(h)=\alpha_h([\mathbf s])\ \forall h\in\mathrm{Stab}$ |
| Orthonormal state | $|\widetilde{[\mathbf s];\chi}\rangle=\sqrt{\frac{|\mathrm{Stab}|}{|G|}}\sum_{g}\chi(g)^*U_g|[\mathbf s]\rangle$ |
| Projection coefficient | $\mathsf{coeff}=\alpha_{g_{\mathbf m}}\chi(g_{\mathbf m})^*$ |
| Matrix element | $H_{\mathbf s',\mathbf s}^\chi=\mathsf{coeff}\sqrt{\frac{|\mathrm{Stab}(\mathbf m)|}{|\mathrm{Stab}(\mathbf s)|}}\delta_{[\mathbf s'],[\mathbf m]}$ |
| Jordan–Wigner sign | $(-1)^{\sum_{k=\min(i,j)+1}^{\max(i,j)-1} n_k}$ |
